In [ ]:
import warnings
warnings.filterwarnings("ignore")
                        
import altair as alt
import folium
import geopandas as gpd
import google.auth
import pandas as pd

import world_cup_vars as wc_vars
import D1_prep_trips as D1
import D2_prep_stop_arrivals as D2
import chart_utils

credentials, _ = google.auth.default()

In [ ]:
sofi_trips = D1.filter_fct_daily_schedule_rt_route_direction_summary_to_special_routes(
    event_name = wc_vars.event_name,
    operator_list = wc_vars.socal_names,
    route_name_dict = wc_vars.special_socal_routes_dict,
    event_time_of_day_dict = wc_vars.sofi_match_times  
)

In [ ]:
sofi_stop_arrivals = D2.filter_fct_daily_scheduled_stops_to_special_routes(
    event_name = wc_vars.event_name,
    operator_list = wc_vars.socal_names,
    route_name_dict = wc_vars.special_socal_routes_dict,
    event_time_of_day_dict = wc_vars.sofi_match_times,
)

In [ ]:
def stop_arrival_change_from_baseline_wide(
    stop_arrivals: gpd.GeoDataFrame
):
    arrivals_by_event_df = D2.aggregate_by_event_type(stop_arrivals)
    
    weekday_wide = D2.make_wide(
        arrivals_by_event_df[arrivals_by_event_df.day_type == "weekday"],
        group_cols = ["schedule_name", "stop_id", "stop_name"],
        metric_cols = ["daily_arrivals"]
    ).rename(columns = {
        **{c: f"weekday_{c}" for c in ["daily_arrivals_event", "daily_arrivals_non_event", "change_daily_arrivals"]}
    })
    
    weekend_wide = D2.make_wide(
        arrivals_by_event_df[arrivals_by_event_df.day_type == "weekend"],
        group_cols = ["schedule_name", "stop_id", "stop_name"],
        metric_cols = ["daily_arrivals"]
    ).rename(columns = {
        **{c: f"weekend_{c}" for c in ["daily_arrivals_event", "daily_arrivals_non_event", "change_daily_arrivals"]}
    })
    
    arrivals_wide = pd.merge(
        weekday_wide,
        weekend_wide,
        on = ["schedule_name", "stop_id", "stop_name"],
        how = "inner"
    ).pipe(D2.merge_in_stop_geom, stop_arrivals)

    arrivals_wide = arrivals_wide.assign(
        combined_change_daily_arrivals = arrivals_wide.weekday_change_daily_arrivals + arrivals_wide.weekend_change_daily_arrivals
    )

    return arrivals_wide

In [ ]:
arrivals_wide = stop_arrival_change_from_baseline_wide(sofi_stop_arrivals)
arrivals_wide.shape

In [ ]:
sofi_stop_arrivals2 = D2.filter_fct_daily_scheduled_stops_to_special_routes_keep_far_stops(
    event_name = wc_vars.event_name,
    operator_list = wc_vars.socal_names,
    route_name_dict = wc_vars.special_socal_routes_dict,
    event_time_of_day_dict = wc_vars.sofi_match_times,
)

sofi_stop_arrivals2.shape

In [ ]:
arrivals_wide2 = stop_arrival_change_from_baseline_wide(
    sofi_stop_arrivals2
)

arrivals_wide2.shape

In [ ]:
arrivals_wide.combined_change_daily_arrivals.max(), arrivals_wide.combined_change_daily_arrivals.min()

In [ ]:
# https://github.com/posit-dev/great-tables/blob/main/great_tables/_data_color/constants.py
SERVICE_CHANGE_COLORS = [
    "#EE6363", #indianred2
    # pick ones from YlGnBu palette and reverse so yellow is most
    #"#081D58", "#225EA8", "#41B6C4", "#C7E9B4", "#FFFF00",
    # pick ones from BuGn palette, but set closer to zero as a light yello
    "#C7E9B4", "#66C2A4", "#41AE76", "#006D2C", "#00441B",
]


YlGnBu_palette = [
    "#FFFFD9",
    "#EDF8B1",
    "#C7E9B4",
    "#7FCDBB",
    "#41B6C4",
    "#1D91C0",
    "#225EA8",
    "#253494",
    "#081D58",
]

BuGn_palette = [
    "#F7FCFD",
    "#E5F5F9",
    "#CCECE6",
    "#99D8C9",
    "#66C2A4",
    "#41AE76",
    "#238B45",
    "#006D2C",
    "#00441B",
]
SERVICE_CHANGE_INDEX = [-25, 0, 25, 50, 100, 150, 200]
SERVICE_CHANGE_CAPTION = "service change (positive = added service; negative = reduced service)"


In [ ]:
# set custom cmap and cutoffs example: PREDICTION_ERROR_COLORS, PREDICTION_ERROR_INDEX
# https://github.com/cal-itp/gtfs-curator/blob/main/rt_predictions/operator_report.ipynb
# change line width thickness example: sofi_service_changes.ipynb
import branca.colormap as cm

m = sofi_trips[["schedule_name", "route_name", "direction_id", "geometry"]].drop_duplicates().explore(
    "route_name",
    tiles = "CartoDB Positron",
    name = "route", legend=False
)

m = arrivals_wide2.explore(
    "combined_change_daily_arrivals",
    m=m,
    cmap = cm.StepColormap(
        colors=SERVICE_CHANGE_COLORS, index=SERVICE_CHANGE_INDEX, 
        vmin=-25, vmax=200, # this is manually here after a check that max is 163
        tick_labels=SERVICE_CHANGE_INDEX,
        caption=SERVICE_CHANGE_CAPTION
    ),
    style_kwds={
        "style_function": lambda x: {
            "radius": x["properties"]["combined_change_daily_arrivals"]*0.05,
            "fillOpacity": 0.5,
            "strokeOpacity": 0.5,
            #"weight":x["properties"]["combined_change_daily_arrivals"]*0.08,
        }
    },
    name = "arrivals"
)

folium.LayerControl().add_to(m)

m

## ideas
Contactless payments, before and after GIS map.
before event, more traffic
over time, how has that changed. more dynamic kind of chart, video?

heatmap, regression charts
* across dates, how it had changed, so it's little cubes, date on x-axis, routes on y-axis
* event on Saturday, compare to other saturdays (6 months or some period vs current Saturday with event)
* buffer ring analysis, near event, what changed 0-500meters, are there more trips, vs 500-1_000 meters, buffer to stop area, see changes to the stop. a buffer-ring analysis, splitting stops into distance bands from the stadium (say 0 to 500 meters, 500 meters to 1 kilometer, 1 to 2 kilometers), would show whether the service boost decays the further you get from the venue.

agency vs agency, specific agency that adds routes during event
mode related analysis, is it easier for bus routes to be added? or train frequencies?
date relative to event, google chart, event date is 0, and other dates are -1, 1, 2, 

bubble map, similar to ring map, but more suited for stops, can show change to bubble map for stops getting near stadiums

## Heatmap

In [ ]:
# heatmap
alt.Chart(sofi_trips).mark_rect().encode(
    y='route_name:O',
    x='service_date:T',
    color='sum(n_trips):Q'
)

In [ ]:
for i in arrivals_by_event_type.schedule_name.unique():
    one_chart = arrivals_by_operator_chart = alt.Chart(arrivals_by_event_type[arrivals_by_event_type.schedule_name==i]).mark_rect().encode(
        y='stop_name:O',
        x='event_day:O',
        color='sum(daily_arrivals):Q',
        tooltip=["sum(daily_arrivals)"]
    ).properties(title = i).interactive()
        
    display(one_chart)

## Bubble Chart by stop
* plot event-baseline (show the change in arrivals by stop)

## Time-of-day comparison


## Statistical Analysis

A control comparison can be used to assess how transit service changes on event days relative to typical service patterns. 

This fits in with comparing daily trips / daily stop arrivals for event vs non-event.
* weekend service is quite different than weekday, so without plotting them separately, we miss out the additional weekend service (since additional service still is much smaller than typical weekday service)

**Difference-in-difference or route-fixed effects**
* that event chart that shows before/event=0/after.
* comparison would be event day vs the last non-event day of similar type
   * Fri event, compared to Thurs (compare 2 weekdays)
   * Sat event, compare to last Sat event (compare 2 Saturdays, or compare to last Sunday?)
* **can compare event vs non-event, near vs far, bus vs rail all at once**!
   * should be able to tease out whether rail or bus added more service
   * think more about the set up for df. what happens if route name changes for special service? need to be able to add them onto the same record to compare what happened.
   * add these boolean columns (is_rail, is_near, is_event). remember to avoid perfect multicollinearity, always 1 less column
   * this might be able to take the entire df, don't need to filter for stops being within certain buffer. so we can take the entirety of those feeds, for all routes, for all stops, and just throw it into this regression, with the right dummy variables.

## Map of Routes for all other feeds
* map of routes (how to compare event vs non-event)?
   * aggregate daily trips on event days (weekday + weekend), aggregate daily trips on non-event days (weekday + weekend)
   * show the difference? 
* can we get stops that show up on routes as a layer too?
* see a chart that filters for that route, show weekday + weekend event vs non-event trips?
* can be combined with World Cup feed

In [ ]:
trips_by_event = sofi_trips.groupby(
    ["schedule_name", "event_day", "day_type", "route_name", "route_type"]
).agg({
    "n_trips": "sum",
    "service_date": "nunique"
}).reset_index().rename(columns = {
    "service_date": "n_days"
})

trips_by_event = trips_by_event.assign(
    daily_trips = trips_by_event.n_trips.divide(trips_by_event.n_days).round(1)
)

In [ ]:
trips_by_event_wide = D2.make_wide(
    trips_by_event,
    group_cols = ["schedule_name", "day_type", "route_name"],
    metric_cols = ["daily_trips"]
)

In [ ]:
alt.Chart(trips_by_event).mark_rect().encode(
    y='route_name:O',
    x='event_day:O',
    color='sum(daily_trips):Q'
)

In [ ]:
selection = alt.selection_point(fields=["schedule_name"], bind="legend")
alt.Chart(trips_by_event).mark_bar().encode(
    x="daily_trips",
    y=alt.Y("event_day", title = ""),
    row=alt.Row("day_type:N", title = ""),
    color=alt.Color("schedule_name:N"), 
    opacity=alt.when(selection).then(alt.value(1)).otherwise(alt.value(0.1)),
).add_params(selection).transform_filter(selection)

In [ ]:
alt.Chart(trips_by_event).mark_bar().encode(
    x="daily_trips",
    y=alt.Y("event_day", title = ""),
    column=alt.Column("day_type:N", title = ""),
    row=alt.Row("schedule_name", title=""),
    color=alt.Color("schedule_name:N"), 
).properties(width=150, height=50)

In [ ]:
selection = alt.selection_point(fields=["schedule_name"], bind="legend")
alt.Chart(trips_by_event_wide).mark_bar().encode(
    x="change_daily_trips",
    y="route_name:N",
    column=alt.Column("day_type:N", title=""),
    color=alt.Color("schedule_name:N"), 
    opacity=alt.when(selection).then(alt.value(1)).otherwise(alt.value(0.1)),
).add_params(selection)